# Mass–spring cloth

*mechanics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/mass_spring_cloth)

A square grid of point masses joined by structural, shear and bending springs, pinned at two corners and released under gravity. An incidence matrix $D_{ea}$ turns the mesh into tensors: one einsum gives every spring vector, another adds up the forces on every node.

**Vector form**

$$m\,\ddot{\mathbf x}_a = \sum_{e\ni a}\mathbf F_e + m\,\mathbf g - m\gamma\,\dot{\mathbf x}_a,\qquad \mathbf F_e = k\,(\ell_e - \ell^0_e)\,\hat{\mathbf u}_e$$

**Index (tensor) form**

$$d_{ei} = D_{ea}x_{ai},\qquad \ell_e = \sqrt{d_{ei}d_{ei}},\qquad \ddot x_{ai} = -\tfrac{k}{m}\,D_{ea}(\ell_e - \ell^0_e)\,\frac{d_{ei}}{\ell_e} + g_i - \gamma\,\dot x_{ai}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{ai}$ | `y[0]` | (N, 3) | position of node $a$; $N = n^2$ masses |
| $D_{ea}$ | `incidence(edges, N)` | (E, N) | incidence matrix: $-1$ / $+1$ at the two ends of spring $e$ |
| $d_{ei} = D_{ea}x_{ai}$ | `einsum('ea,ai->ei', D, x)` | (E, 3) | repeated $a$ ⇒ summed: vector along every spring |
| $\ell_e$ | `sqrt(einsum('ei,ei->e', d, d))` | (E,) | current spring lengths; $\ell^0_e$ are the rest lengths |
| $F_{ai}$ | `-einsum('ea,e,ei->ai', D, tension, u)` | (N, 3) | repeated $e$ ⇒ summed: net spring force on node $a$ |
| $F_{ai} (sparse)$ | `F.index_add_(0, i, f); F.index_add_(0, j, -f)` | (N, 3) | the same sum as a scatter over the edge list |
| $E$ | `energy(traj, D, rest, cloth)` | (T,) | kinetic + elastic + gravitational energy |

Shear springs along the diagonals stop the squares collapsing into rhombi, and springs that skip one node give the cloth some bending stiffness. The two pinned corners simply get zero acceleration. Air damping $\gamma$ is the only loss, so the total energy can only go down ($dE/dt = -m\gamma\lvert\mathbf v\rvert^2$) and with $\gamma = 0$ it is conserved. The springs are stiff, so the time step is tied to $\sqrt{m/k}$. Self-collisions are ignored. Each node has mass 10 g and the grid spacing is 10 cm.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/mass_spring_cloth/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n, spacing, node_mass = params["n_side"], 0.1, 0.01
    x0 = tensors.grid(n, spacing, dev)  # (N, 3)
    edges = tensors.springs(n, dev)  # (E, 2)
    D = tensors.incidence(edges, n * n)  # (E, N)
    rest = tensors.rest_lengths(x0, edges)  # (E,)
    free = tensors.free_mask(n, dev)  # (N,)
    cloth = tensors.parameters(params["k"], params["damping"], node_mass, 9.81, dev)

    dt = min(0.002, 0.15 * math.sqrt(node_mass / params["k"]))  # stable RK4 step for the stiffest spring
    times, traj = solve.solve(x0, D, rest, free, cloth, params["t_max"], dt)
    E = equation.energy(traj, D, rest, cloth)  # (T,)

    print(f"{n * n} masses, {len(edges)} springs, dt = {dt * 1e3:.2f} ms, {len(times) - 1} steps")
    print(f"energy: start {E[0]:.4f} J, end {E[-1]:.4f} J")
    return plot.animate(times, traj, E, n, out_dir / "mass_spring_cloth.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/mass_spring_cloth/equation.py
r"""Physics → tensor translation.

A cloth as N point masses (nodes a) joined by E springs (edges e), hanging under gravity with air damping γ:

    m \ddot{\mathbf x}_a = \sum_{\text{springs } e \ni a} \mathbf F_e + m \mathbf g - m \gamma \dot{\mathbf x}_a .

The incidence matrix D_{ea} encodes the mesh: spring e from node i to node j has D_{ei} = −1, D_{ej} = +1.
With it, everything is an index contraction (repeated indices summed; i ∈ {x, y, z}):

    d_{ei} = D_{ea} x_{ai}                         edge vector of every spring, sum over nodes a
    \ell_e = \sqrt{d_{ei} d_{ei}},   u_{ei} = d_{ei} / \ell_e
    F_{ai} = -D_{ea} \, k (\ell_e - \ell^0_e) \, u_{ei}     Hooke's law, sum over springs e

    \ddot x_{ai} = F_{ai} / m + g_i - \gamma \dot x_{ai}        (zero for pinned nodes)

Big meshes store D sparsely; the same sum over e is then a scatter: index_add_ over the edge list.
The energy  E = ½ m \dot x_{ai}\dot x_{ai} + ½ k (\ell_e - \ell^0_e)^2 - m g_i x_{ai}  only decreases when γ > 0
(dE/dt = −m γ \dot x_{ai}\dot x_{ai}) and is conserved when γ = 0.
"""

import torch


def edge_vectors(x: torch.Tensor, D: torch.Tensor) -> torch.Tensor:
    r"""d_{ei} = D_{ea} x_{ai}.   x: (N, 3), D: (E, N) → (E, 3)"""
    return torch.einsum("ea,ai->ei", D, x)


def spring_forces(x: torch.Tensor, D: torch.Tensor, rest: torch.Tensor, k: float) -> torch.Tensor:
    r"""F_{ai} = -D_{ea} k (\ell_e - \ell^0_e) u_{ei}.   → (N, 3)"""
    d = edge_vectors(x, D)  # (E, 3)
    length = torch.sqrt(torch.einsum("ei,ei->e", d, d))  # (E,)
    tension = k * (length - rest)  # (E,) positive when stretched
    u = d / length[:, None]  # (E, 3) unit vectors i → j
    return -torch.einsum("ea,e,ei->ai", D, tension, u)


def spring_forces_scatter(x: torch.Tensor, edges: torch.Tensor, rest: torch.Tensor, k: float) -> torch.Tensor:
    """Same forces from the edge list (E, 2) with gather + index_add_ — how large meshes do it."""
    i, j = edges[:, 0], edges[:, 1]
    d = x[j] - x[i]  # (E, 3)
    length = d.norm(dim=-1)  # (E,)
    f = (k * (length - rest) / length)[:, None] * d  # (E, 3) pull on node i towards j
    F = torch.zeros_like(x)
    F.index_add_(0, i, f)  # each spring pulls its first node towards the second …
    F.index_add_(0, j, -f)  # … and the second node back (Newton's third law)
    return F


def rhs(y: torch.Tensor, D: torch.Tensor, rest: torch.Tensor, free: torch.Tensor, cloth: dict) -> torch.Tensor:
    """dy/dt for y of shape (2, N, 3): y[0] = positions, y[1] = velocities. free: (N,) 1 = moves, 0 = pinned."""
    x, v = y[0], y[1]
    a = spring_forces(x, D, rest, cloth["k"]) / cloth["m"] + cloth["g"] - cloth["gamma"] * v  # (N, 3)
    a = free[:, None] * a  # pinned nodes never accelerate
    return torch.stack([v, a])


def energy(y: torch.Tensor, D: torch.Tensor, rest: torch.Tensor, cloth: dict) -> torch.Tensor:
    r"""E = ½ m \dot x_{ai}\dot x_{ai} + ½ k (\ell_e − \ell^0_e)^2 − m g_i x_{ai}.   y: (T, 2, N, 3) → (T,)"""
    x, v = y[:, 0], y[:, 1]  # (T, N, 3)
    kinetic = 0.5 * cloth["m"] * torch.einsum("tai,tai->t", v, v)
    d = torch.einsum("ea,tai->tei", D, x)  # (T, E, 3)
    stretch = d.norm(dim=-1) - rest  # (T, E)
    elastic = 0.5 * cloth["k"] * (stretch**2).sum(-1)
    gravity = -cloth["m"] * torch.einsum("i,tai->t", cloth["g"], x)
    return kinetic + elastic + gravity


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/mass_spring_cloth/tensors.py
"""Build the concrete tensors: node positions, the spring list, the incidence matrix and the pins.

Node (row r, column c) of an n × n grid has index a = r·n + c and starts flat at (c·h, r·h, 0).
float64 keeps the energy bookkeeping clean.
"""

import torch

DTYPE = torch.float64


def grid(n: int, spacing: float, device: torch.device | None = None) -> torch.Tensor:
    """x_ai of shape (N = n², 3): a flat horizontal sheet."""
    r, c = torch.meshgrid(torch.arange(n), torch.arange(n), indexing="ij")  # (n, n) each
    x = torch.stack([c * spacing, r * spacing, torch.zeros(n, n)], dim=-1)  # (n, n, 3)
    return x.reshape(n * n, 3).to(DTYPE).to(device)


def springs(n: int, device: torch.device | None = None) -> torch.Tensor:
    """Edge list (E, 2) of node pairs: structural (neighbours), shear (diagonals) and bend (two apart) springs."""
    ids = torch.arange(n * n).reshape(n, n)  # ids[r, c] = r·n + c
    pairs = [
        (ids[:, :-1], ids[:, 1:]),  # horizontal neighbours
        (ids[:-1, :], ids[1:, :]),  # vertical neighbours
        (ids[:-1, :-1], ids[1:, 1:]),  # diagonal ↘
        (ids[:-1, 1:], ids[1:, :-1]),  # diagonal ↙ (stops the cloth shearing flat)
        (ids[:, :-2], ids[:, 2:]),  # two apart, horizontally (resists bending)
        (ids[:-2, :], ids[2:, :]),  # two apart, vertically
    ]
    edges = torch.cat([torch.stack([a.flatten(), b.flatten()], dim=-1) for a, b in pairs])
    return edges.to(device)


def incidence(edges: torch.Tensor, n_nodes: int) -> torch.Tensor:
    """D_ea of shape (E, N): −1 at the first node of spring e, +1 at the second, 0 elsewhere."""
    D = torch.zeros(len(edges), n_nodes, dtype=DTYPE, device=edges.device)
    e = torch.arange(len(edges), device=edges.device)
    D[e, edges[:, 0]] = -1.0
    D[e, edges[:, 1]] = 1.0
    return D


def rest_lengths(x0: torch.Tensor, edges: torch.Tensor) -> torch.Tensor:
    """ℓ⁰_e of shape (E,): every spring is relaxed in the starting configuration."""
    return (x0[edges[:, 1]] - x0[edges[:, 0]]).norm(dim=-1)


def free_mask(n: int, device: torch.device | None = None) -> torch.Tensor:
    """(N,) with 1 for moving nodes and 0 for the two pinned corners of the first row."""
    free = torch.ones(n * n, dtype=DTYPE, device=device)
    free[0] = 0.0  # corner (0, 0)
    free[n - 1] = 0.0  # corner (0, n − 1)
    return free


def parameters(k: float, gamma: float, node_mass: float, g: float, device: torch.device | None = None) -> dict:
    """Material constants in one dict; g_i points down the z axis."""
    gravity = torch.tensor([0.0, 0.0, -g], dtype=DTYPE, device=device)  # (3,)
    return {"k": k, "gamma": gamma, "m": node_mass, "g": gravity}


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/mass_spring_cloth/solve.py
"""Step the whole cloth forward with 4th-order Runge–Kutta, starting from rest."""

import torch

from problems._common.integrators import integrate

from .equation import rhs


def solve(
    x0: torch.Tensor, D: torch.Tensor, rest: torch.Tensor, free: torch.Tensor, cloth: dict, t_max: float, dt: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), trajectory (T, 2, N, 3)) with positions in [:, 0] and velocities in [:, 1].

    The springs make the system stiff: dt must stay well below √(m/k), the period of one mass on one spring.
    """
    n_steps = int(round(t_max / dt))
    y0 = torch.stack([x0, torch.zeros_like(x0)])  # (2, N, 3) released from rest
    traj = integrate(lambda y, t: rhs(y, D, rest, free, cloth), y0, dt, n_steps)
    times = torch.arange(n_steps + 1, device=x0.device, dtype=x0.dtype) * dt
    return times, traj


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/mass_spring_cloth/plot.py
"""3D animation of the falling cloth as a shaded surface, with its total energy beside it."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(times: torch.Tensor, traj: torch.Tensor, E: torch.Tensor, n: int, path: Path) -> Path:
    t = times.cpu().numpy()  # (T,)
    x = traj[:, 0].cpu().numpy().reshape(len(t), n, n, 3)  # (T, n, n, 3) node (r, c) → position
    energy = E.cpu().numpy()  # (T,)
    lo, hi = x.reshape(-1, 3).min(0), x.reshape(-1, 3).max(0)

    fig = plt.figure(figsize=(7, 4))
    ax_cloth = fig.add_subplot(1, 2, 1, projection="3d")
    ax_energy = fig.add_subplot(1, 2, 2)

    # --- left: the cloth ---
    ax_cloth.set_xlim(lo[0], hi[0])
    ax_cloth.set_ylim(lo[1], hi[1])
    ax_cloth.set_zlim(lo[2], hi[2])
    ax_cloth.set_box_aspect(hi - lo + 1e-9)  # true proportions
    ax_cloth.set_xlabel("x (m)", fontsize=8)
    ax_cloth.set_ylabel("y (m)", fontsize=8)
    ax_cloth.set_zlabel("z (m)", fontsize=8)
    ax_cloth.tick_params(labelsize=7)
    for axis in (ax_cloth.xaxis, ax_cloth.yaxis, ax_cloth.zaxis):
        axis.set_major_locator(plt.MaxNLocator(3))
    pins = x[0, 0, [0, n - 1]]  # (2, 3) the pinned corners never move
    ax_cloth.plot(pins[:, 0], pins[:, 1], pins[:, 2], "k^", ms=6)
    clock = ax_cloth.text2D(0.02, 0.95, "", transform=ax_cloth.transAxes, fontsize=9)
    surface = [None]  # the surface is redrawn every frame

    # --- right: total energy (only goes down with damping) ---
    ax_energy.plot(t, energy, lw=1, color="#d1d5db")
    (line,) = ax_energy.plot([], [], lw=1.6, color=ACCENT[1])
    ax_energy.set_xlim(t[0], t[-1])
    ax_energy.set_xlabel("t (s)")
    ax_energy.set_ylabel("total energy (J)")

    def update(step: int):
        if surface[0] is not None:
            surface[0].remove()
        X, Y, Z = x[step, ..., 0], x[step, ..., 1], x[step, ..., 2]
        surface[0] = ax_cloth.plot_surface(X, Y, Z, color="#93c5fd", edgecolor="#1e40af", lw=0.4, shade=True)
        line.set_data(t[: step + 1], energy[: step + 1])
        clock.set_text(f"t = {t[step]:.2f} s")

    ax_cloth.view_init(elev=15, azim=-60)
    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 80), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_side': 12,
    'k': 200.0,
    'damping': 0.3,
    't_max': 4.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.mass_spring_cloth")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.mass_spring_cloth.main")
problem_dir = Path("problems/mechanics/mass_spring_cloth")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
